In [3]:
import sys
print(sys.version)

In [3]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [ ]:
!pip install orekit-jpype

In [22]:
import orekit_jpype as orekit
orekit.initVM()

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

print(os.listdir())

In [ ]:
from orekit_jpype.pyhelpers import setup_orekit_data

setup_orekit_data("orekit-data-main.zip")

In [ ]:
from org.orekit.time import TimeScalesFactory

utc = TimeScalesFactory.getUTC()

print("UTC:", utc)
print("SUCCESS: Orekit data loaded correctly!")

In [29]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from org.orekit.frames import FramesFactory
from org.orekit.time import AbsoluteDate, TimeScalesFactory
from org.orekit.utils import Constants
from org.orekit.orbits import KeplerianOrbit, PositionAngleType
from org.orekit.propagation.analytical import KeplerianPropagator

In [31]:
utc = TimeScalesFactory.getUTC()

initial_date = AbsoluteDate(
    2026,
    1,
    1,
    12,
    0,
    0.0,
    utc
)

frame = FramesFactory.getEME2000()

orbit = KeplerianOrbit(
    7000000.0,                 # Semi-major axis (m)
    0.001,                     # Eccentricity
    math.radians(98.0),        # Inclination
    math.radians(0.0),         # Argument of Perigee
    math.radians(0.0),         # RAAN
    math.radians(0.0),         # True Anomaly
    PositionAngleType.TRUE,
    frame,
    initial_date,
    Constants.WGS84_EARTH_MU
)

In [32]:
propagator = KeplerianPropagator(orbit)

In [34]:
times = []

x = []
y = []
z = []

vx = []
vy = []
vz = []

for t in range(0, 5401, 60):

    current_date = initial_date.shiftedBy(float(t))

    state = propagator.propagate(current_date)

    pv = state.getPVCoordinates()

    pos = pv.getPosition()
    vel = pv.getVelocity()

    times.append(t)

    x.append(pos.getX()/1000)
    y.append(pos.getY()/1000)
    z.append(pos.getZ()/1000)

    vx.append(vel.getX()/1000)
    vy.append(vel.getY()/1000)
    vz.append(vel.getZ()/1000)

In [ ]:
df = pd.DataFrame({
    "Time (s)": times,
    "X (km)": x,
    "Y (km)": y,
    "Z (km)": z,
    "VX (km/s)": vx,
    "VY (km/s)": vy,
    "VZ (km/s)": vz
})

df.head()

In [ ]:
plt.figure(figsize=(8,8))
plt.plot(df["X (km)"], df["Y (km)"])
plt.scatter(0,0,s=250,label="Earth")
plt.xlabel("X (km)")
plt.ylabel("Y (km)")
plt.title("Ground Truth Orbit")
plt.grid(True)
plt.axis("equal")
plt.legend()
plt.show()

In [ ]:
df.to_csv("synthetic_orbit_dataset.csv", index=False)